In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

SEED = 42
TEST_SIZE = 0.2
DATA = "../data/train.csv"
DROP = ["PassengerId", "Name", "Ticket", "Cabin", "SibSp"]

df = pd.read_csv(DATA).drop(columns=DROP)
X, y = df.drop(columns="Survived"), df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

In [ ]:
num_cols = ["Age", "Fare", "Parch", "Pclass"]
cat_cols = ["Sex", "Embarked"]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),                 
    ]), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh",  OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)),
    ]), cat_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

model = Pipeline([
    ("prep", preprocess),
    ("mlp",  MLPClassifier(
        hidden_layer_sizes=(32, 16),      
        activation="relu",
        solver="adam",
        alpha=1e-3,                        
        learning_rate_init=1e-3,
        batch_size=32,
        max_iter=500,
        early_stopping=True,               
        validation_fraction=0.2,
        n_iter_no_change=20,
        random_state=42,
    )),
])
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=3))
print("Test AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("Итераций:", model.named_steps["mlp"].n_iter_)
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0      0.780     0.873     0.824       110
           1      0.750     0.609     0.672        69

    accuracy                          0.771       179
   macro avg      0.765     0.741     0.748       179
weighted avg      0.769     0.771     0.765       179

Test AUC: 0.8377
Итераций: 35
[[96 14]
 [27 42]]


In [ ]:
import sys, os
sys.path.append(os.path.abspath("../.."))
from utils import log_result

log_result(
    model_name="MLP",
    y_true=y_test,
    y_proba=y_proba,      
    y_pred=y_pred,       
    params={"activation": "relu", "solver": "adam", "hidden_layer_sizes": "32, 16"},
    notes=""
)

✓ MLP: AUC=0.8377, acc=0.7709 → c:\Users\АуфПК\VSCode\titanic\results.csv


{'model': 'MLP',
 'timestamp': '2026-11-04 00:36',
 'auc': 0.8377,
 'accuracy': 0.7709,
 'f1': 0.672,
 'precision': 0.75,
 'recall': 0.6087,
 'params': '{}',
 'notes': ''}